In [2]:
### RAG PIPELINE- data ingestion to vector database pipeline

import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [3]:
### read the pdf file and load it as a document
def process_pdf(pdf_directory):
    '''process pdf file in the dir'''

    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_file = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_file)} PDF files in the directory.")

    for pdf_file in pdf_file:
        print(f"Processing file: {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
        
            for doc in documents:
                doc.metadata["source"] = pdf_file.name
                doc.metadata["file_path"] = "pdf"

            all_documents.extend(documents)
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    return all_documents

all_documents = process_pdf("../data/pdf")

Found 2 PDF files in the directory.
Processing file: ..\data\pdf\Advancing_Genetic_Engineering_through_AI_Sequencin.pdf
Processing file: ..\data\pdf\BIOLOGY_USING_AI.pdf


In [3]:
all_documents

[Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': 'Advancing_Genetic_Engineering_through_AI_Sequencin.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'file_path': 'pdf'}, page_content='Advancing Genetic Engineering Through AI: Sequencing \nand Editing Innovations \nShili Sun1,a,* \n1Department of Economics, University of Washington, Seattle, United States \na. ssl0226@asu.edu.pl \n*corresponding author \nAbstract: As an important part of modern biotechnology, genetic engineering is widely used \nin fields such as disease treatment, agricultural improvement, and environmental protection. \nGene sequencing technology, especially next -generation sequencing technology, prov ides a \npowerful tool for studying biological genetic information. However, with the rapid growth of \ngenomic data, how to efficiently and accurately analyze and apply this huge data has become \na major challenge facing genetic engineering. In recent years, a rtificial 

In [4]:
### text splitting into chuncks 

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    '''split documents into chunks'''
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, separators=["\n\n", "\n", " ", ""], length_function=len)
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")
    
    if split_docs:
        print(f"Example of a split document chunk:\n{split_docs[0].page_content[:500]}...")  # Print the first 500 characters of the first chunk
    
    return split_docs

In [5]:
chunks = split_documents(all_documents)

Split 45 documents into 211 chunks.
Example of a split document chunk:
Advancing Genetic Engineering Through AI: Sequencing 
and Editing Innovations 
Shili Sun1,a,* 
1Department of Economics, University of Washington, Seattle, United States 
a. ssl0226@asu.edu.pl 
*corresponding author 
Abstract: As an important part of modern biotechnology, genetic engineering is widely used 
in fields such as disease treatment, agricultural improvement, and environmental protection. 
Gene sequencing technology, especially next -generation sequencing technology, prov ides a 
power...


In [6]:
### embedding and vector database storage will be the next steps

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb 
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    def __init__(self, model_name:str = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        '''load the sentence transformer model'''
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded embedding model: {self.model_name}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
             Generate embeddings for a list of texts
        
            Args:
            texts: List of text strings to embed
            
            Returns:
                numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded. Call load_model() first.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
embeddingmanager = EmbeddingManager()
embeddingmanager

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5241.23it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedding model: all-MiniLM-L6-v2


In [8]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 211


In [9]:
# Reset the current collection so only current PDFs are indexed
try:
    vectorstore.client.delete_collection(vectorstore.collection_name)
except Exception:
    pass

vectorstore.collection = vectorstore.client.get_or_create_collection(
    name=vectorstore.collection_name,
    metadata={"description": "PDF document embeddings for RAG"}
)
print(f"Collection reset complete. Current count: {vectorstore.collection.count()}")

Collection reset complete. Current count: 0


In [10]:
# Ingest chunks into vector store (run this once after creating chunks)
texts = [chunk.page_content for chunk in chunks]
embeddings = embeddingmanager.generate_embeddings(texts)

vectorstore.add_documents(chunks, embeddings)
print(f"Collection now has {vectorstore.collection.count()} chunks")

Generating embeddings for 211 texts...


Batches: 100%|██████████| 7/7 [00:10<00:00,  1.44s/it]


Generated embeddings with shape: (211, 384)
Adding 211 documents to vector store...
Successfully added 211 documents to vector store
Total documents in collection: 211
Collection now has 211 chunks


In [ ]:
chunks

[Document(metadata={'producer': 'pypdf', 'creator': 'PyPDF', 'creationdate': '', 'source': 'Advancing_Genetic_Engineering_through_AI_Sequencin.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1', 'file_path': 'pdf'}, page_content='Advancing Genetic Engineering Through AI: Sequencing \nand Editing Innovations \nShili Sun1,a,* \n1Department of Economics, University of Washington, Seattle, United States \na. ssl0226@asu.edu.pl \n*corresponding author \nAbstract: As an important part of modern biotechnology, genetic engineering is widely used \nin fields such as disease treatment, agricultural improvement, and environmental protection. \nGene sequencing technology, especially next -generation sequencing technology, prov ides a \npowerful tool for studying biological genetic information. However, with the rapid growth of \ngenomic data, how to efficiently and accurately analyze and apply this huge data has become \na major challenge facing genetic engineering. In recent years, a rtificial 

In [11]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        if self.vector_store.collection.count() == 0:
            print("Vector store is empty. Run the ingestion cell first.")
            return []

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                if retrieved_docs:
                    print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
                else:
                    top_scores = [round(1 - d, 4) for d in distances]
                    print("Documents were found, but all were filtered by score_threshold.")
                    print(f"Top similarity scores: {top_scores}")
                    print("Try score_threshold=0.0 or 0.1")
            else:
                print("No documents found in vector search.")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embeddingmanager)
rag_retriever

In [14]:
rag_retriever.retrieve("What is genetic engineering?", top_k=3, score_threshold=0.0)

Retrieving documents for query: 'What is genetic engineering?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.92it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_321e8fc2_0',
  'content': 'Advancing Genetic Engineering Through AI: Sequencing \nand Editing Innovations \nShili Sun1,a,* \n1Department of Economics, University of Washington, Seattle, United States \na. ssl0226@asu.edu.pl \n*corresponding author \nAbstract: As an important part of modern biotechnology, genetic engineering is widely used \nin fields such as disease treatment, agricultural improvement, and environmental protection. \nGene sequencing technology, especially next -generation sequencing technology, prov ides a \npowerful tool for studying biological genetic information. However, with the rapid growth of \ngenomic data, how to efficiently and accurately analyze and apply this huge data has become \na major challenge facing genetic engineering. In recent years, a rtificial intelligence (AI) \ntechnology, especially deep learning, has been widely used in the automated processing of \nlarge-scale data and has shown great potential in genetic engineering. AI techno

In [15]:
rag_retriever.retrieve("Why ai is used in bio", top_k=3, score_threshold=0.0)

Retrieving documents for query: 'Why ai is used in bio'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.04it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_38d42864_60',
  'content': 'reduced waste, increased output, and time to market. Machine learning and deep -learning-based smart \nprograms can modify metabolic pathways of living systems to achieve optimal outputs with minimal \ninputs. This article summarizes the potential of AI in various fields of biology, including medicine, \nagriculture, and bio-based industry. The article highlights the potential of AI in improving human quality \nof life and enhancing the efficiency of bio-based industrial setups[10]. \n. \n3. Significance and Scope of AI in Biology: \nArtificial intelligence (AI) refers to computerized systems that work and react in ways that require \nintelligence. AI technologies and methodologies are used in various biological sciences and biology \nR&D, including engineering biology [11]. AI has enabled advances in research and development across \nvarious industries, such as analyzing genomic data to determine the genetic basis of traits and potentially \nun

In [14]:
### integration vector db context pipeline w/ LLM output

import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found. Add it to your .env file.")

llm = ChatGroq(
    api_key=groq_api_key,
    model="llama-3.1-8b-instant",
    temperature=0.7,
    max_tokens=1024,
)


def rag_simple(query, retriever, llm, top_k=3, score_threshold=0.0):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=score_threshold)
    context = "\n\n".join([doc["content"] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(prompt)
    return response.content if hasattr(response, "content") else str(response)

In [15]:
rag_simple("Why is AI used in genetic engineering?", rag_retriever, llm, top_k=3, score_threshold=0.0)

Retrieving documents for query: 'Why is AI used in genetic engineering?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 45.13it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


'AI is used in genetic engineering to improve efficiency, accuracy, and automation of data analysis, predict gene function, and optimize experimental processes, among other applications. It helps in optimizing guide RNA design, variation detection, genome association analysis, and gene editing, ultimately enhancing the precision and speed of genetic engineering research.'

In [18]:
### enhanced RAG pipeline features

def rag_advanced(query, retrieve, llm, top_k=5, min_score=0.2, return_context=False):
    results = retrieve(query, top_k=top_k, score_threshold=min_score)
    
    if not results:
        return "No relevant documents found to answer the question."

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    prompt = f"""Use the following context to answer the question concisely.\n{context}\n\nQuestion: {query}\n\nAnswer:"""

    response = llm.invoke(prompt.format(context=context, query=query))

    output = {
        'answer': response.content if hasattr(response, "content") else str(response),
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['retrieved_context'] = context
    return output
res = rag_advanced("What are the applications of AI in genetic engineering?", rag_retriever.retrieve, llm, top_k=5, min_score=0.0, return_context=True)
print("ANSWER:", res['answer'])
print("\nSOURCES:")
for source in res['sources']:
    print(f"- {source['source']} (Page: {source['page']}, Score: {source['score']:.4f})")
    print(f"  Preview: {source['preview']}\n")
    

Retrieving documents for query: 'What are the applications of AI in genetic engineering?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.44it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


ANSWER: The applications of AI in genetic engineering include:

1. Improving the efficiency, accuracy, and automation of data analysis.
2. Predicting gene function.
3. Optimizing guide RNA design of CRISPR-Cas9 systems to improve the precision and efficiency of gene editing.
4. Variation detection of genetic data and genome association analysis to help identify genetic variants associated with specific diseases.
5. Automating sample processing, data analysis, and interpretation of results in laboratory settings.
6. Analyzing and applying large-scale genomic data efficiently and accurately.
7. Improving the speed and quality of scientific research in genetic engineering.

SOURCES:
- Advancing_Genetic_Engineering_through_AI_Sequencin.pdf (Page: 1, Score: 0.3726)
  Preview: While AI technology brings many conveniences and advances, it also faces a series of complex 
challenges. On the technical side, AI models are often limited by poor data quality, and lack of 
transparency. In terms of 

In [21]:
### Advanced RAG pipeline : streaming, citation, history, multi-turn, etc.
from typing import Dict, Any


class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        return_context: bool = False,
        stream: bool = False,
        summarize: bool = False,
    ) -> Dict[str, Any]:
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)

        if not results:
            return {
                "question": question,
                "answer": "No relevant documents found to answer the question.",
                "sources": [],
                "confidence": 0.0,
                "summary": None,
                "history": self.history,
            }

        context = "\n\n".join([doc["content"] for doc in results])
        sources = [
            {
                "source": doc["metadata"].get("source_file", doc["metadata"].get("source", "unknown")),
                "page": doc["metadata"].get("page", "unknown"),
                "score": doc["similarity_score"],
                "preview": doc["content"][:300] + "...",
            }
            for doc in results
        ]
        confidence = max([doc["similarity_score"] for doc in results])

        prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {question}

Answer:"""

        response = self.llm.invoke(prompt)
        answer = response.content if hasattr(response, "content") else str(response)

        if stream:
            print("streaming answer:")
            for i in range(0, len(answer), 80):
                print(answer[i:i + 80], end="", flush=True)
            print()

        citations = [f"[{i + 1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke(summary_prompt)
            summary = summary_resp.content if hasattr(summary_resp, "content") else str(summary_resp)

        self.history.append(
            {
                "question": question,
                "answer": answer,
                "sources": sources,
                "confidence": confidence,
            }
        )

        output = {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "confidence": confidence,
            "summary": summary,
            "history": self.history,
        }

        if return_context:
            output["retrieved_context"] = context

        return output


adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
res = adv_rag.query(
    "What are the applications of AI in Pharmacy list 10 factors affecting this field of study as per 20 marks?",
    top_k=5,
    min_score=0.0,
    return_context=True,
    summarize=True,
)
print("ANSWER:", res["answer"])
print("\nSOURCES:")
for source in res["sources"]:
    print(f"- {source['source']} (Page: {source['page']}, Score: {source['score']:.4f})")
    print(f"  Preview: {source['preview']}\n")
if res["summary"]:
    print("SUMMARY:", res["summary"])

Retrieving documents for query: 'What are the applications of AI in Pharmacy list 10 factors affecting this field of study as per 20 marks?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:04<00:00,  4.45s/it]


Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
ANSWER: **Applications of AI in Pharmacy:**

1. **Biomarker and target identification**: AI can help identify potential biomarkers and targets for disease treatment.
2. **Indication prioritization**: AI can help prioritize indications for drug development based on market potential and unmet medical needs.
3. **Drug-like molecule design**: AI can design new drug-like molecules with desired properties.
4. **Pharmacokinetics prediction**: AI can predict the pharmacokinetics of new drugs, reducing the need for animal testing.
5. **Drug-target interaction**: AI can predict the interaction between drugs and their targets, reducing the risk of adverse effects.
6. **Clinical trial design**: AI can help design more efficient and effective clinical trials.
7. **Predictive analytics**: AI can analyze large datasets to predict patient outcomes and identify high-risk patients.
8. **Personalized medicine**: AI can help